# Rutinaa Merge
En este proto no se toma en cuenta a los demas proveedoeres debido  aue solo serta enfocado a exel del norte

# Librerias

In [1]:
# DIRECTOB2B/ETL/rutina_merge.py
import os
import sys
import time
import json
import pickle
import logging
import gc
import unicodedata
import re
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from dotenv import load_dotenv

# MANIPULACIÓN DE DATOS
import pandas as pd
import numpy as np

# BASE DE DATOS
from sqlalchemy import create_engine, text
from sqlalchemy.orm import sessionmaker, Session
from psycopg2.extras import execute_batch

# RED
import requests

# COnfiguracion

## Variables Entorno

In [2]:
# CONFIGURACIÓN
load_dotenv()  # Carga variables de entorno

True

In [3]:
# LOGGING
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger(__name__)

## Base de Datos

In [4]:
# --- CONFIGURACIÓN DB ---
def get_db_engine():
    try:
        user = os.getenv("DB_USER")
        password = os.getenv("DB_PASS")
        host = os.getenv("DB_HOST")
        port = os.getenv("DB_PORT")
        database = os.getenv("DB_NAME")
        
        if not all([user, password, host, port, database]):
            raise ValueError("Faltan variables de entorno para la BD")

        engine = create_engine(f'postgresql://{user}:{password}@{host}:{port}/{database}')
        return engine
    except Exception as e:
        logger.error(f"Error conectando a BD: {e}")
        return None

# Funciones

## Normaliza Texto

In [5]:

# --- UTILIDADES ---
def normalize_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = unicodedata.normalize('NFKD', text).encode('ASCII', 'ignore').decode('ASCII')
    text = re.sub(r'(\d+)\s*(ml|cm|mm|kg|g)', r'\1_\2', text)
    text = re.sub(r'[^a-z0-9\s\-_\/]', ' ', text)
    return text.strip()

In [6]:
def normalizar_marca(marca: str) -> str:
    return re.sub(r'\s+', ' ', str(marca).strip()).upper()

## Divisora

In [7]:
def divisora_producto_detalle(df):
    # --- 1. TABLA DE PRODUCTOS ---
    columnas_finales_productos = ['csku', 'cnombre', 'cmarca', 'cdescripcion', 'cespecificaciones', 'cimagen', 'bestatus']
    
    # Extraemos solo las columnas que sí existen en el DF original para evitar KeyError
    cols_existentes = [col for col in columnas_finales_productos if col in df.columns]
    df_tbl_productos = df[cols_existentes].copy()

    # Validar y rellenar columnas requeridas si faltan
    columnas_requeridas = ["csku", "cnombre", "cmarca", "cdescripcion", "cespecificaciones", "cimagen", "tcreate_at", "tupdate_at", "bestatus"]

    for col in columnas_requeridas:
        if col not in df_tbl_productos.columns:
            if col in ["tcreate_at", "tupdate_at"]:
                df_tbl_productos[col] = datetime.now()
            else:
                df_tbl_productos[col] = None


    # --- 2. TABLA DE PRECIOS / DETALLE (SOLO EXEL) ---
    # Creamos el DataFrame directamente con las columnas de Exel
    df_precios_filtrado = pd.DataFrame({
        'csku': df['csku'],
        # Pongo un respaldo por si la columna viene como ID_PROVEEDOR_exel o solo ID_PROVEEDOR
        'nid_proveedor': df.get('ID_PROVEEDOR_exel', df.get('ID_PROVEEDOR')), 
        'ndisponibilidad': df.get('disponibilidad_exel'),
        'cmoneda': df.get('moneda_exel'),
        'nprecio': df.get('precio_exel'),
        'cclave_producto': df.get('clave_producto_exel')
    })

    # Eliminar filas donde el ID_PROVEEDOR es nulo (reemplaza el antiguo dropna de la concatenación)
    df_precios_filtrado = df_precios_filtrado.dropna(subset=['nid_proveedor']).copy()

    # Filtrar solo los precios cuyos SKUs realmente existen en la tabla de productos
    df_precios_filtrado = df_precios_filtrado[df_precios_filtrado['csku'].isin(df_tbl_productos['csku'])]

    # Normalizar la moneda
    if 'cmoneda' in df_precios_filtrado.columns:
        df_precios_filtrado['cmoneda'] = df_precios_filtrado['cmoneda'].replace({'Pesos': 'MXN', 'Dolares': 'USD'})

    return df_tbl_productos, df_precios_filtrado

## icecat

## Categorizador

## icECAT

In [ ]:
# ==========================================================
# Función principal para actualizar fichas Icecat
# ==========================================================
def actualizar_fichas_icecat():
    """
    Obtiene productos sin ficha Icecat, consulta la API en paralelo,
    y guarda resultados en la base de datos omitiendo errores individuales.
    """
    
    # ==========================
    # 1. Obtener productos sin ficha
    # ==========================
    engine = conection_bd()
    if not engine:
        return

    try:
        query = "SELECT csku, cmarca FROM tbl_producto WHERE cficha_icecat IS NULL;"
        df_productos = pd.read_sql(query, engine)
        print(f"📦 Catálogo obtenido: {len(df_productos)} productos sin ficha.")
    except Exception as e:
        print("❌ Error al ejecutar la consulta inicial:", e)
        engine.dispose()
        return

    # ==========================
    # 2. Configuración API Icecat
    # ==========================
    username = os.getenv("ICECAT_USERNAME")
    app_key = os.getenv("ICECAT_APP_KEY")
    language = "es"
    base_url = "https://live.icecat.biz/api"

    resultados = []
    contador_exitos = 0
    errores_api = 0

    # ==========================
    # 3. Función de llamada API
    # ==========================
    def hacer_llamada(idx, product_code, brand):
        # Validación de marca
        if pd.isna(brand) or str(brand).strip() == "":
            print(f"⚠️ {idx} - {product_code} omitido: Marca vacía")
            return None

        params = {
            "UserName": username,
            "Language": language,
            "ProductCode": product_code,
            "Brand": brand,
            "app_key": app_key
        }

        try:
            response = requests.get(base_url, params=params, timeout=10)

            if response.status_code == 200:
                try:
                    data = response.json()
                    print(f"✅ {idx} - {product_code} ({brand}) OK")
                    return {"sku": product_code, "data": data}
                except ValueError:
                    print(f"❌ {idx} - {product_code} ({brand}) Error: JSON inválido")
            else:
                print(f"❌ {idx} - {product_code} ({brand}) Error HTTP {response.status_code}")
                
        except requests.RequestException as e:
            print(f"❌ {idx} - {product_code} ({brand}) Excepción de red: {e}")

        return None

    # ==========================
    # 4. Ejecutar llamadas en paralelo
    # ==========================
    print("🚀 Iniciando consultas a la API de Icecat...")
    tasks = [
        (idx, row['csku'], row['cmarca'])
        for idx, row in df_productos.iterrows()
        if pd.notna(row['csku'])
    ]

    with ThreadPoolExecutor(max_workers=10) as executor:
        future_to_data = {
            executor.submit(hacer_llamada, idx, sku, marca): idx
            for idx, sku, marca in tasks
        }

        for future in as_completed(future_to_data):
            result = future.result()
            if result:
                resultados.append(result)
                contador_exitos += 1
            else:
                errores_api += 1

    # ==========================
    # 5. Actualizar base de datos iterando los resultados directamente
    # ==========================
    if not resultados:
        print("⚠️ No se obtuvieron resultados exitosos para actualizar.")
        engine.dispose()
        return

    print("💾 Iniciando guardado en base de datos...")
    conn = engine.raw_connection()
    # Habilitar autocommit para que un error en un UPDATE no aborte la transacción completa
    conn.autocommit = True 
    cursor = conn.cursor()

    query_update = "UPDATE tbl_producto SET cficha_icecat = %s WHERE csku = %s;"
    errores_sql = 0

    for r in resultados:
        sku = r["sku"]
        
        # ---------------------------------------------------------
        # Limpieza exhaustiva de caracteres nulos
        # ---------------------------------------------------------
        json_str = json.dumps(r["data"], ensure_ascii=False)
        
        # 1. Elimina el carácter nulo invisible (\x00)
        json_limpio = json_str.replace('\x00', '')
        
        # 2. Elimina el texto literal "\u0000"
        json_limpio = json_limpio.replace('\\u0000', '')
        # ---------------------------------------------------------

        try:
            cursor.execute(query_update, (json_limpio, sku))
        except Exception as e:
            print(f"❌ Error SQL al actualizar SKU {sku}: {e}")
            errores_sql += 1
            continue  # Ignora este error específico y pasa al siguiente producto

    # Cerrar conexiones
    cursor.close()
    conn.close()
    engine.dispose()

    # ==========================
    # 6. Resumen de ejecución
    # ==========================
    print("\n==================================")
    print("✅ Actualización completada")
    print(f"📊 Total procesados con éxito: {contador_exitos}")
    print(f"❌ Total errores API omitidos: {errores_api}")
    print(f"⚠️  Total errores SQL omitidos: {errores_sql}")
    print("==================================")

In [8]:
def categorizador():
    """Ejecuta la inferencia de la red neuronal para categorizar productos."""
    try:
        import tensorflow as tf
        from tensorflow.keras.utils import pad_sequences
    except ImportError:
        logger.error("Tensorflow no instalado.")
        return

    logger.info("Iniciando proceso de categorización neuronal...")
    engine = get_db_engine()
    if not engine: return

    try:
        # Obtener productos sin categoría
        query = """
            SELECT tp.csku, tp.cnombre, tp.cdescripcion, tp.cmarca, tnc.nid as id_subcategoria
            FROM tbl_producto AS tp
            LEFT JOIN tbl_subcategoria AS tnc ON tnc.nid = tp.nid_subcategoria
            WHERE tp.nid_subcategoria IS NULL
        """
        df_prod = pd.read_sql(query, engine)
        
        if df_prod.empty:
            logger.info("No hay productos pendientes de categorización.")
            return

        # Rutas dinámicas
        BASE_DIR = os.path.dirname(os.path.abspath(__file__))
        RN_DIR = os.path.join(BASE_DIR, "Red_neuronal")
        
        path_model = os.path.join(RN_DIR, "modelo_categorias_optimizado.keras")
        path_tok = os.path.join(RN_DIR, "tokenizer.pkl")
        path_enc = os.path.join(RN_DIR, "labelencoder.pkl")

        if not os.path.exists(path_model):
            raise FileNotFoundError(f"Modelo no encontrado: {path_model}")

        # Cargar artefactos
        model = tf.keras.models.load_model(path_model)
        with open(path_tok, "rb") as f: tokenizer = pickle.load(f)
        with open(path_enc, "rb") as f: le = pickle.load(f)

        # Preprocesamiento
        df_prod["texto"] = (
            df_prod["cnombre"].fillna("") + " " + 
            df_prod["cdescripcion"].fillna("") + " " + 
            df_prod["cmarca"].fillna("")
        )
        df_prod["texto"] = df_prod["texto"].apply(normalize_text)
        
        seq = tokenizer.texts_to_sequences(df_prod["texto"])
        X = pad_sequences(seq, maxlen=200)

        # Inferencia
        preds = model.predict(X, verbose=0)
        y_classes = np.argmax(preds, axis=1)
        categorias = le.inverse_transform(y_classes)
        df_prod["categoria_predicha"] = categorias

        # Obtener IDs de subcategorías
        df_sub = pd.read_sql("SELECT nid_subcategoria, nombre_subcategoria FROM tbl_nueva_subcategoria", engine)
        
        df_final = df_prod.merge(df_sub, left_on="categoria_predicha", right_on="nombre_subcategoria", how="left")
        
        # Update en Batch
        updates = list(zip(df_final["id_subcategoria_y"], df_final["csku"]))
        updates = [(int(cat), sku) for cat, sku in updates if pd.notna(cat)]

        if updates:
            with engine.connect() as conn:
                with conn.connection.cursor() as cursor:
                    execute_batch(cursor, "UPDATE tbl_producto SET nid_subcategoria  = %s WHERE csku = %s", updates)
                    conn.connection.commit()
            logger.info(f"Categorizados {len(updates)} productos.")

    except Exception as e:
        logger.error(f"Error en categorizador: {e}")
    finally:
        engine.dispose()
        tf.keras.backend.clear_session()
        gc.collect()

## Estatus Productos

In [9]:
def actualizar_estatus_productos():
    engine = get_db_engine()
    if not engine: return
    try:
        with engine.begin() as conn:
            # Desactivar
            conn.execute(text("""
                UPDATE tbl_producto SET bestatus = 'f' 
                WHERE ndisponibilidad_total = 0 OR csku NOT IN (SELECT DISTINCT csku FROM tbl_detalle_producto) or ndisponibilidad_total is NULL
            """))
            # Activar
            conn.execute(text("""
                UPDATE tbl_producto SET bestatus = 't' 
                WHERE ndisponibilidad_total > 0
            """))
            logger.info("Estatus de productos actualizado.")
    finally:
        engine.dispose()

## PRecio

In [10]:
def ponderacion_de_precio():
    engine = get_db_engine()
    if not engine: 
        return
    
    try:

        # tipo de cambio
        df_div = pd.read_sql(
            "SELECT divisa, precio FROM tbl_cambio_divisas", 
            engine
        )
        tc = dict(zip(df_div['divisa'], df_div['precio']))

        # precios proveedores
        query = """
            SELECT tdp.csku, tdp.cmoneda, tdp.nprecio, tdp.ndisponibilidad
            FROM tbl_detalle_producto tdp
            WHERE tdp.nprecio > 0 
            AND tdp.ndisponibilidad > 0
        """
        df = pd.read_sql(query, engine)

        # convertir a MXN
        df['precio_mxn'] = df['cmoneda'].map(tc).fillna(1) * df['nprecio']

        resultados = []

        for sku, g in df.groupby('csku'):

            precios = g['precio_mxn'].values
            disponibilidad = g['ndisponibilidad'].values

            mu = precios.mean()
            sigma = precios.std()

            # evitar división por 0
            if sigma == 0:
                sigma = mu * 0.05

            # peso gaussiano
            peso_gauss = np.exp(-((precios - mu) ** 2) / (2 * sigma ** 2))

            # combinar con disponibilidad
            peso_final = peso_gauss * disponibilidad

            costo = np.sum(precios * peso_final) / np.sum(peso_final)

            # agregar 5% extra
            costo = float(round(costo * 1.05, 2))

            disponibilidad_total = int(disponibilidad.sum())

            resultados.append((costo, disponibilidad_total, sku))

        # actualizar DB
        with engine.raw_connection() as conn:
            cursor = conn.cursor()

            execute_batch(
                cursor,
                """
                UPDATE tbl_producto 
                SET nprecio_b2b = %s,
                    ndisponibilidad_total = %s
                WHERE csku = %s
                """,
                resultados
            )

            conn.commit()

        logger.info("Ponderación gaussiana de precios finalizada.")

    except Exception as e:
        logger.error(f"Error en ponderación: {e}")

    finally:
        engine.dispose()


## Actualizador de Antiguos Productos

In [11]:
def actualizar_catalogos_db(df_old: pd.DataFrame, df_old_price: pd.DataFrame) -> None:
    """
    Actualiza las tablas tbl_producto y tbl_detalle_producto en la base de datos
    utilizando execute_batch para alto rendimiento.
    """
    engine = get_db_engine()
    if not engine: return
    try:
        # 1. Limpieza de datos: PostgreSQL no entiende np.nan, necesita None (NULL)
        # Esto es crítico porque 'cespecificaciones' y 'cimagen' tienen valores nulos en tu df_old
        df_prod_clean = df_old.replace({np.nan: None})
        df_price_clean = df_old_price.replace({np.nan: None})

        # 2. Conversión a lista de diccionarios para psycopg2
        datos_producto = df_prod_clean.to_dict('records')
        datos_precio = df_price_clean.to_dict('records')

        # 3. Query condicional para tbl_producto (Solo actualiza si hay cambios reales)
        # Utilizamos IS DISTINCT FROM para manejar comparaciones con NULLs de forma segura
        query_producto = """
            UPDATE tbl_producto
            SET 
                cnombre = %(cnombre)s,
                cmarca = %(cmarca)s,
                cdescripcion = %(cdescripcion)s,
                cespecificaciones = %(cespecificaciones)s,
                cimagen = %(cimagen)s,
                tupdate_at = CURRENT_TIMESTAMP
            WHERE csku = %(csku)s
            AND (
                cnombre IS DISTINCT FROM %(cnombre)s OR
                cmarca IS DISTINCT FROM %(cmarca)s OR
                cdescripcion IS DISTINCT FROM %(cdescripcion)s OR
                cespecificaciones IS DISTINCT FROM %(cespecificaciones)s OR
                cimagen IS DISTINCT FROM %(cimagen)s
            );
        """

        # 4. Query incondicional para tbl_detalle_producto (Actualiza sí o sí)
        query_precio = """
            UPDATE tbl_detalle_producto
            SET 
                ndisponibilidad = %(ndisponibilidad)s,
                cmoneda = %(cmoneda)s,
                nprecio = %(nprecio)s
            WHERE csku = %(csku)s;
        """

        # 5. Ejecución transaccional en lote (Batch)
        logger.info("Iniciando actualización en lote hacia la base de datos...")
        
        # Extraemos la conexión raw de psycopg2 desde el engine de SQLAlchemy
        with engine.connect() as conn:
            raw_conn = conn.connection
            try:
                with raw_conn.cursor() as cur:
                    # Actualizar tbl_producto
                    logger.info(f"Procesando {len(datos_producto)} registros para tbl_producto...")
                    execute_batch(cur, query_producto, datos_producto, page_size=1000)
                    
                    # Actualizar tbl_detalle_producto
                    logger.info(f"Procesando {len(datos_precio)} registros para tbl_detalle_producto...")
                    execute_batch(cur, query_precio, datos_precio, page_size=1000)
                
                # Si todo sale bien, hacemos commit de la transacción
                raw_conn.commit()
                logger.info("¡Actualización completada y confirmada en la base de datos!")
                
            except Exception as e:
                # Si hay un error (ej. llave foránea rota, timeout), hacemos rollback
                raw_conn.rollback()
                logger.error(f"Error durante la actualización. Se aplicó Rollback. Detalle: {e}")
                raise
    finally:
        engine.dispose()

## Actualizar  DOlar

In [12]:
def actualizar_tipo_cambio_usd():
    # Token y URL Banxico
    token = "3da738eaf30e07518304fea87b5d610f7e2e16f6fa917a3c61b7ad5a3cdcd861"
    url = "https://www.banxico.org.mx/SieAPIRest/service/v1/series/SF43718/datos/oportuno"
    headers = {"Bmx-Token": token}

    # Consulta a la API
    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        data = response.json()
        try:
            dato = float(data['bmx']['series'][0]['datos'][0]['dato'])
            precio = int(dato) + (dato != int(dato))  # Ajuste del tipo de cambio
            fecha_actualizacion = datetime.now()

            engine = get_db_engine()

            update_sql = """
            UPDATE tbl_cambio_divisas
            SET 
                precio = :precio,
                fehca_actualizacion = :fecha_actualizacion
            WHERE divisa = :divisa;
            """

            insert_sql = """
            INSERT INTO tbl_cambio_divisas (divisa, precio, fehca_actualizacion)
            VALUES (:divisa, :precio, :fecha_actualizacion);
            """

            with engine.begin() as conn:
                result = conn.execute(text(update_sql), {
                    "precio": precio,
                    "fecha_actualizacion": fecha_actualizacion,
                    "divisa": "USD"
                })

                if result.rowcount == 0:
                    conn.execute(text(insert_sql), {
                        "divisa": "USD",
                        "precio": precio,
                        "fecha_actualizacion": fecha_actualizacion
                    })
                    print("✅ Registro insertado.")
                else:
                    print("✅ Registro actualizado.")

            engine.dispose()

        except Exception as e:
            print("⚠️ Error al procesar:", e)
    else:
        print("❌ Error al obtener datos:", response.status_code, response.text)

# Obtener tipos de cambio /PARA PRECIO PONDERADO
def obtener_tipos_cambio(engine):
    """
    Obtiene los tipos de cambio de monedas a MXN desde la tabla `tbl_cambio_divisas`.

    Parámetros:
    -----------
    engine : sqlalchemy.engine.base.Engine
        Conexión activa a la base de datos.

    Retorna:
    --------
    dict
        Diccionario con claves como códigos de moneda (e.g. 'USD', 'EUR') y valores de tipo de cambio (float).
    """
    query = "SELECT divisa, precio FROM tbl_cambio_divisas;"
    df_cambio = pd.read_sql(query, engine)
    tipos_cambio = dict(zip(df_cambio['divisa'], df_cambio['precio']))
    return tipos_cambio


# Main

In [13]:
logger.info("INICIANDO RUTINA MERGE")
    
engine = get_db_engine()

2026-03-09 17:21:47,709 - INFO - INICIANDO RUTINA MERGE


In [14]:
 # 1. Cargar DataFrames directamente a variables
tablas = ['temp_tbl_exel']

for t in tablas:
    try:
        # Leemos el SQL y lo asignamos dinámicamente al nombre de la tabla
        globals()[t] = pd.read_sql(f"SELECT * FROM {t}", engine)
        logger.info(f"Cargado {t}: {len(globals()[t])} registros")
    except Exception as e:
        globals()[t] = pd.DataFrame()
        logger.error(f"Error al cargar {t}: {e}")

2026-03-09 17:21:50,703 - INFO - Cargado temp_tbl_exel: 14106 registros


In [15]:
temp_tbl_exel.info()

<class 'pandas.DataFrame'>
RangeIndex: 14106 entries, 0 to 14105
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   SKU                    14106 non-null  str    
 1   nombre_exel            14106 non-null  str    
 2   disponibilidad_exel    14106 non-null  int64  
 3   precio_exel            14106 non-null  float64
 4   moneda_exel            14106 non-null  str    
 5   marca_exel             14106 non-null  str    
 6   categoria_exel         14106 non-null  str    
 7   especificaciones_exel  14106 non-null  str    
 8   imagen_exel            14106 non-null  str    
 9   clave_producto_exel    14106 non-null  str    
 10  ID_PROVEEDOR           14106 non-null  int64  
 11  descripcion_exel       14106 non-null  str    
dtypes: float64(1), int64(2), str(9)
memory usage: 1.3 MB


In [16]:
df_final = temp_tbl_exel.copy()

In [17]:
# Renombrar a esquema DB
mapping = {
    'SKU': 'csku', 'nombre_exel': 'cnombre', 'categoria_exel': 'ccategoria',
    'marca_exel': 'cmarca', 'descripcion': 'cdescripcion', 
    'especificaciones_exel': 'cespecificaciones', 'imagen_exel': 'cimagen', 'descripcion_exel':'cdescripcion'
}
df_final.rename(columns=mapping, inplace=True)

In [18]:
 # Limpieza
df_final.dropna(subset=['csku', 'cnombre'], inplace=True)
df_final = df_final[df_final['cnombre'] != 'ND']
df_final['bestatus'] = 't'


In [19]:
df_final = df_final.replace(['NULL', 'null', 'None', 'nan'], np.nan)

In [20]:
# Normalización final de marcas
df_final['cmarca'] = df_final['cmarca'].apply(normalizar_marca)

In [21]:
df_final.shape

(14106, 13)

In [22]:
 # Deduplicación por SKU (Optimización: Agrupar y tomar el primero válido)
df_final = df_final.groupby('csku', as_index=False).first()


In [23]:
df_final.shape

(14098, 13)

In [24]:
df_final

,csku,cnombre,disponibilidad_exel,precio_exel,moneda_exel,cmarca,ccategoria,cespecificaciones,cimagen,clave_producto_exel,ID_PROVEEDOR,cdescripcion,bestatus
0,00-0614-00-8,Cafetera Koblenz CKM-1350 en Color Negro con A...,61,2438.7272,MXN,KOBLENZ,Electrónica de Consumo,NaN,NaN,KBLCAFAB003,4,Cafetera Koblenz CKM-1350 en Color Negro con A...,t
1,00-1522-00-2,Regulador Koblenz RI-20 1900VA / 1400W Entrad...,151,831.5216,MXN,KOBLENZ,Energía y Cables,"[{'grupo': 'Características Generales:', 'cara...",['https://contenidos.exel.com.mx/ImagenesProdu...,KBCREUAB042,4,Regulador Koblenz RI-20 1900VA / 1400W Entrad...,t
2,00-1560-2,Regulador Koblenz ER-2550 2500VA/2000W 6 Conta...,161,818.4800,MXN,KOBLENZ,Energía y Cables,"[{'grupo': 'Características Generales:', 'cara...",['https://contenidos.exel.com.mx/ImagenesProdu...,KBCREUAB002,4,Protegen la vida de sus aparatos electrónicos ...,t
3,00-1588-3,Regulador Koblenz ER-2250 2250VA/1000W 8 Conta...,86,672.8280,MXN,KOBLENZ,Energía y Cables,"[{'grupo': 'Características Generales:', 'cara...",['https://contenidos.exel.com.mx/ImagenesProdu...,KBCREUAB025,4,Regulador Koblenz ER-2250 2250VA/1000W 8 Conta...,t
4,00-1596-00-6,Regulador Koblenz RI-2002 2000VA/1500W 1 Contacto,19,1281.5192,MXN,KOBLENZ,Energía y Cables,"[{'grupo': 'Características Generales:', 'cara...",['https://contenidos.exel.com.mx/ImagenesProdu...,KBCREUAB029,4,Regulador Koblenz RI-2002 2000VA/1500W 1 Contacto,t
...,...,...,...,...,...,...,...,...,...,...,...,...,...
14093,esc-12AC,Bolígrafo Samsill Escolar Tipo Gel Punto Fino ...,0,0.0000,MXN,SAMSILL,Oficina y Escolar,NaN,NaN,SSPBOLAB006,4,Bolígrafo Samsill Escolar Tipo Gel Punto Fino ...,t
14094,esc-12NC,Bolígrafo Samsill Escolar Tipo Gel Punto Fino ...,20,65.6604,MXN,SAMSILL,Oficina y Escolar,"[{'grupo': 'Características Generales:', 'cara...",['https://contenidos.exel.com.mx/ImagenesProdu...,SSPBOLAB007,4,Bolígrafo Samsill Escolar Tipo Gel Punto Fino ...,t
14095,esc-12RC,Bolígrafo Samsill Escolar Tipo Gel Punto Fino ...,23,65.6604,MXN,SAMSILL,Oficina y Escolar,"[{'grupo': 'Características Generales:', 'cara...",['https://contenidos.exel.com.mx/ImagenesProdu...,SSPBOLAB008,4,Bolígrafo Samsill Escolar Tipo Gel Punto Fino ...,t
14096,sam-12AC,Bolígrafo Samsill Punto Mediano 1.0 mm Color A...,1,23.3792,MXN,SAMSILL,Oficina y Escolar,"[{'grupo': 'Características Generales:', 'cara...",['https://contenidos.exel.com.mx/ImagenesProdu...,SSPBOLAB003,4,Bolígrafo Samsill Punto Mediano 1.0 mm Color A...,t


In [25]:
# 3. Separar Master / Detalle
df_prod, df_det = divisora_producto_detalle(df_final)

In [26]:
df_prod.shape

(14098, 9)

In [27]:
df_det.shape

(14098, 6)

In [28]:
df_prod.sample(2)

,csku,cnombre,cmarca,cdescripcion,cespecificaciones,cimagen,bestatus,tcreate_at,tupdate_at
4289,714075,Barra PDU Intellinet Multicontactos 15 Montaje...,INTELLINET,Los PDUs (Unidades de Distribución de Energía)...,"[{'grupo': 'Características Generales:', 'cara...",['https://contenidos.exel.com.mx/ImagenesProdu...,t,2026-03-09 17:21:52.216492,2026-03-09 17:21:52.219921
8395,GFS-10V,GAFETE MAE SOLIDO VERTICAL C/50,MAE,GAFETE MAE SOLIDO VERTICAL C/50,"[{'grupo': 'Detalle tecníco', 'caracteristicas...",['https://contenidos.exel.com.mx/ImagenesProdu...,t,2026-03-09 17:21:52.216492,2026-03-09 17:21:52.219921


In [29]:
df_det.info()

<class 'pandas.DataFrame'>
RangeIndex: 14098 entries, 0 to 14097
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   csku             14098 non-null  str    
 1   nid_proveedor    14098 non-null  int64  
 2   ndisponibilidad  14098 non-null  int64  
 3   cmoneda          14098 non-null  str    
 4   nprecio          14098 non-null  float64
 5   cclave_producto  14098 non-null  str    
dtypes: float64(1), int64(2), str(3)
memory usage: 661.0 KB


In [30]:
 # 4. Obtener Productos Existentes en BD
existing_skus = pd.read_sql("SELECT csku FROM tbl_producto", engine)['csku'].tolist()
existing_skus_set = set(existing_skus)

In [31]:
# Crear máscara una sola vez (más eficiente)
mask_new = ~df_prod['csku'].isin(existing_skus_set)

# Separar productos
df_new = df_prod[mask_new]
df_old = df_prod[~mask_new]

# Separar detalle precio usando la misma lógica
mask_new_price = df_det['csku'].isin(df_new['csku'])

df_new_price = df_det[mask_new_price]
df_old_price = df_det[~mask_new_price]

In [32]:
df_old_price.shape

(14098, 6)

In [33]:
# Insertar nuevos productos
if not df_new.empty:
    df_new.to_sql('tbl_producto', engine, if_exists='append', index=False)
    logger.info(f"Insertados {len(df_new)} nuevos productos.")

In [34]:
# Insertar precios de productos nuevos
if not df_new_price.empty:
    df_new_price.to_sql('tbl_detalle_producto', engine, if_exists='append', index=False)
    logger.info(f"Insertados {len(df_new_price)} nuevos detalles de precio.")

In [35]:
actualizar_catalogos_db(df_old,df_old_price)

2026-03-09 17:21:54,140 - INFO - Iniciando actualización en lote hacia la base de datos...
2026-03-09 17:21:55,092 - INFO - Procesando 14098 registros para tbl_producto...
2026-03-09 17:22:01,572 - INFO - Procesando 14098 registros para tbl_detalle_producto...
2026-03-09 17:22:03,785 - INFO - ¡Actualización completada y confirmada en la base de datos!


In [36]:
print("Actualizando Precio del Dolar")
actualizar_tipo_cambio_usd()

Actualizando Precio del Dolar
✅ Registro actualizado.


In [37]:
# 8. Post-Procesos
ponderacion_de_precio()

2026-03-09 17:22:19,987 - INFO - Ponderación gaussiana de precios finalizada.


In [38]:
actualizar_estatus_productos()

2026-03-09 17:22:21,356 - INFO - Estatus de productos actualizado.


In [39]:
# 9. ML
categorizador()

2026-03-09 17:23:16,499 - INFO - Iniciando proceso de categorización neuronal...
2026-03-09 17:23:18,217 - ERROR - Error en categorizador: name '__file__' is not defined
2026-03-09 17:23:20,419 - WARNING - From c:\Users\MoisesEugenioNavaMar\Github\Excel-del-Norte\.venv\Lib\site-packages\keras\src\backend\common\global_state.py:82: The name tf.reset_default_graph is deprecated. Please use tf.compat.v1.reset_default_graph instead.



In [ ]:
pip install sklearn